# DFT Validation — Notebook B (Mg + Ga + Fe + references)

**Runs 21 QE SCF calculations:** undoped (2) + Li metal (1) + Mg (6) + Ga (6) + Fe (6)

**Run this AFTER Notebook A completes.**

**Estimated runtime:** ~6-8 hours on Colab Pro (single core, ecutwfc=60 Ry)

In [ ]:
# Cell 1: Install Quantum ESPRESSO
!apt-get -qq update
!apt-get -qq install -y quantum-espresso
!which pw.x && echo 'QE installed successfully'

In [ ]:
# Cell 2: Mount Google Drive and set up directories
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

DRIVE_DIR = '/content/drive/MyDrive/dft_sqs_validation'
WORK_DIR = '/content/dft_work_B'

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/pseudo', exist_ok=True)
os.makedirs(f'{WORK_DIR}/outputs', exist_ok=True)

# Copy pseudopotentials
for f in os.listdir(f'{DRIVE_DIR}/pseudo'):
    if f.endswith('.UPF'):
        shutil.copy2(f'{DRIVE_DIR}/pseudo/{f}', f'{WORK_DIR}/pseudo/{f}')
        print(f'  Copied {f}')

# Copy QE input files for this notebook's batch
BATCH_B = [
    'undoped_lith', 'undoped_delith', 'li_metal',
    'Mg_ord_lith', 'Mg_ord_delith',
    'Mg_sqs0_lith', 'Mg_sqs0_delith',
    'Mg_sqs1_lith', 'Mg_sqs1_delith',
    'Ga_ord_lith', 'Ga_ord_delith',
    'Ga_sqs0_lith', 'Ga_sqs0_delith',
    'Ga_sqs1_lith', 'Ga_sqs1_delith',
    'Fe_ord_lith', 'Fe_ord_delith',
    'Fe_sqs0_lith', 'Fe_sqs0_delith',
    'Fe_sqs1_lith', 'Fe_sqs1_delith',
]

for name in BATCH_B:
    src = f'{DRIVE_DIR}/qe_sqs_inputs/{name}.in'
    dst = f'{WORK_DIR}/{name}.in'
    shutil.copy2(src, dst)
    print(f'  Copied {name}.in')

shutil.copy2(f'{DRIVE_DIR}/qe_sqs_inputs/manifest.json', f'{WORK_DIR}/manifest.json')

print(f'\nReady: {len(BATCH_B)} calculations to run')

In [ ]:
# Cell 3: Fix paths + patch QE parameters for production quality
#
# Fixes applied (from self-critique):
#   1. pseudo_dir -> local work dir
#   2. ecutwfc: 40 -> 60 Ry (40 too low for TM oxides with PAW)
#   3. ecutrho: 320 -> 480 Ry (8x ecutwfc)
#   4. mixing_beta: 0.1 -> 0.35 (0.1 too slow, caused timeouts)
#   5. electron_maxstep: 500 -> 200 (sufficient with higher mixing)
#   6. tprnfor: .false. -> .true. (print forces to verify MACE geometry)
#   7. degauss: 0.02 -> 0.01 (0.02 too large for semiconductors)
#   8. Add startingwfc/startingpot for better initial guess

import glob
import os

for inp_file in glob.glob(f'{WORK_DIR}/*.in'):
    with open(inp_file, 'r') as f:
        content = f.read()
    
    # Path fix
    content = content.replace("pseudo_dir = './pseudo'", f"pseudo_dir = '{WORK_DIR}/pseudo'")
    
    # Cutoff: 40 -> 60 Ry (MAJOR fix)
    content = content.replace('ecutwfc = 40.0', 'ecutwfc = 60.0')
    content = content.replace('ecutrho = 320.0', 'ecutrho = 480.0')
    
    # Convergence speed
    content = content.replace('mixing_beta = 0.1', 'mixing_beta = 0.35')
    content = content.replace('electron_maxstep = 500', 'electron_maxstep = 200')
    
    # Print forces to check MACE geometry quality
    content = content.replace('tprnfor = .false.', 'tprnfor = .true.')
    
    # Reduce smearing
    content = content.replace('degauss = 0.02', 'degauss = 0.01')
    
    # Add better initial guess if not present
    if 'startingwfc' not in content:
        content = content.replace(
            "  conv_thr = 1e-05",
            "  conv_thr = 1e-05\n  startingwfc = 'atomic+random'\n  startingpot = 'atomic'"
        )
    
    with open(inp_file, 'w') as f:
        f.write(content)

print('Patched all input files:')
print('  pseudo_dir  -> local work dir')
print('  ecutwfc     -> 60.0 Ry (was 40.0)')
print('  ecutrho     -> 480.0 Ry (was 320.0)')
print('  mixing_beta -> 0.35 (was 0.1)')
print('  electron_maxstep -> 200 (was 500)')
print('  tprnfor     -> .true. (was .false.) — forces will be printed')
print('  degauss     -> 0.01 (was 0.02)')
print('  Added startingwfc/startingpot')
print()
print('NOTE: Cell parameters are frozen at ideal LiCoO2 (not MACE-relaxed).')
print('This is a known limitation — acknowledged in paper.')

# Clear old failed checkpoints
ckpt = f'{DRIVE_DIR}/results/checkpoint_B.json'
if os.path.exists(ckpt):
    os.remove(ckpt)
    print(f'\nCleared old checkpoint: {ckpt}')
else:
    print('\nNo old checkpoint to clear')

# Verify one file looks correct
print('\n--- Verification (first input file) ---')
test_file = f'{WORK_DIR}/undoped_lith.in'
if os.path.exists(test_file):
    with open(test_file) as f:
        for line in f:
            line = line.strip()
            if any(kw in line for kw in ['ecutwfc', 'ecutrho', 'mixing_beta', 'degauss', 'tprnfor', 'pseudo_dir', 'electron_maxstep']):
                print(f'  {line}')

In [ ]:
# Cell 4: Run all QE calculations WITH per-calc checkpointing to Drive
#
# RESUME-SAFE: If Colab disconnects, re-run Cells 1-4.
# Skips completed calcs automatically.
#
# Uses single-core pw.x (mpirun hangs on Colab).

import subprocess
import time
import json
import shutil
import os
import re

CHECKPOINT_DIR = f'{DRIVE_DIR}/results'
CHECKPOINT_FILE = f'{CHECKPOINT_DIR}/checkpoint_B.json'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Load existing checkpoint (resume after disconnect) ──
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        results = json.load(f)
    completed = [k for k, v in results.items() if v.get('converged') is not None]
    print(f'RESUMING: Found checkpoint with {len(completed)} completed calcs')
    print(f'  Already done: {completed}')
else:
    results = {}
    print('Starting fresh (no checkpoint found)')

failed = [k for k, v in results.items() if v.get('converged') == False]
t_total_start = time.time()

def extract_max_force(stdout_text):
    """Extract total force from QE output (requires tprnfor=.true.)"""
    max_force = None
    for line in stdout_text.split('\n'):
        if 'Total force' in line:
            match = re.search(r'Total force\s*=\s*([\d.]+)', line)
            if match:
                max_force = float(match.group(1))
                max_force *= 25.7112
    return max_force

for i, name in enumerate(BATCH_B):
    if name in results and results[name].get('converged') is not None:
        status = '✓' if results[name]['converged'] else '✗'
        print(f'[{i+1}/{len(BATCH_B)}] {name} — {status} SKIPPED (in checkpoint)')
        continue

    print(f'\n[{i+1}/{len(BATCH_B)}] Running {name}...', flush=True)
    t_start = time.time()
    
    inp_file = f'{WORK_DIR}/{name}.in'
    out_file = f'{WORK_DIR}/outputs/{name}.out'
    
    try:
        with open(inp_file) as fin:
            result = subprocess.run(
                ['pw.x'],
                stdin=fin,
                capture_output=True, text=True,
                timeout=7200,
                cwd=WORK_DIR
            )
        
        with open(out_file, 'w') as f:
            f.write(result.stdout)
            if result.stderr:
                f.write('\n--- STDERR ---\n')
                f.write(result.stderr)
        
        if 'convergence has been achieved' in result.stdout:
            for line in result.stdout.split('\n'):
                if '!' in line and 'total energy' in line:
                    energy_ry = float(line.split('=')[1].split('Ry')[0].strip())
                    energy_ev = energy_ry * 13.605698
                    max_force = extract_max_force(result.stdout)
                    results[name] = {
                        'energy_Ry': energy_ry,
                        'energy_eV': energy_ev,
                        'converged': True,
                        'time_s': time.time() - t_start,
                        'max_force_eV_per_Ang': max_force,
                    }
                    force_str = f', F_max={max_force:.3f} eV/Å' if max_force else ''
                    print(f'  ✓ E = {energy_ry:.6f} Ry ({time.time()-t_start:.0f}s{force_str})')
                    if max_force and max_force > 0.1:
                        print(f'  ⚠ Large residual force — MACE geometry may be poor')
                    break
        else:
            elapsed = time.time() - t_start
            print(f'  ✗ DID NOT CONVERGE ({elapsed:.0f}s)')
            for l in result.stdout.strip().split('\n')[-5:]:
                print(f'    {l}')
            failed.append(name)
            results[name] = {'converged': False, 'time_s': elapsed}
            
    except subprocess.TimeoutExpired:
        print(f'  ✗ TIMEOUT (>2 hours)')
        failed.append(name)
        results[name] = {'converged': False, 'time_s': 7200, 'error': 'timeout'}
    except Exception as e:
        print(f'  ✗ ERROR: {e}')
        failed.append(name)
        results[name] = {'converged': False, 'error': str(e)}
    
    # ── CHECKPOINT to Drive after EVERY calc ──
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(results, f, indent=2)
    if os.path.exists(out_file):
        shutil.copy2(out_file, f'{CHECKPOINT_DIR}/{name}.out')
    
    n_done = sum(1 for v in results.values() if v.get('converged') is not None)
    n_ok = sum(1 for v in results.values() if v.get('converged') == True)
    print(f'  💾 Saved to Drive: {n_ok} converged, {n_done}/{len(BATCH_B)} total')

t_total = time.time() - t_total_start
n_converged = sum(1 for v in results.values() if v.get('converged') == True)
print(f'\n{"="*60}')
print(f'DONE: {n_converged}/{len(BATCH_B)} converged in {t_total/60:.1f} min')
if failed:
    print(f'FAILED: {failed}')

print(f'\n--- MACE Geometry Quality (residual DFT forces) ---')
for name, r in results.items():
    if r.get('converged') and r.get('max_force_eV_per_Ang'):
        flag = '⚠' if r['max_force_eV_per_Ang'] > 0.1 else '✓'
        print(f'  {flag} {name:25s}: F_max = {r["max_force_eV_per_Ang"]:.4f} eV/Å')

print(f'\nCheckpoint: {CHECKPOINT_FILE}')

In [ ]:
# Cell 5: Results are already on Google Drive (saved by Cell 4 checkpointing)

import json
print(f'Results saved at: {CHECKPOINT_FILE}')
print(f'QE outputs saved at: {CHECKPOINT_DIR}/*.out')
print()
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        r = json.load(f)
    n_ok = sum(1 for v in r.values() if v.get('converged'))
    print(f'Converged: {n_ok}/{len(r)}')

In [ ]:
# Cell 6: Quick sanity check — undoped voltage + reference cross-check
if 'undoped_lith' in results and 'undoped_delith' in results and 'li_metal' in results:
    if all(results[k]['converged'] for k in ['undoped_lith', 'undoped_delith', 'li_metal']):
        E_lith = results['undoped_lith']['energy_eV']
        E_delith = results['undoped_delith']['energy_eV']
        E_li = results['li_metal']['energy_eV'] / 2
        n_li = 18
        V = -(E_delith - E_lith + n_li * E_li) / n_li
        print(f'Undoped LiCoO2 DFT voltage: {V:.3f} V')
        print(f'Expected (experiment): ~3.9 V')
        print(f'\nCross-check: compare these reference energies with Notebook A.')
        print(f'  undoped_lith:   {results["undoped_lith"]["energy_Ry"]:.8f} Ry')
        print(f'  undoped_delith: {results["undoped_delith"]["energy_Ry"]:.8f} Ry')
        print(f'  li_metal:       {results["li_metal"]["energy_Ry"]:.8f} Ry')
        print(f'Both notebooks should give IDENTICAL values for these three.')